# Generator Demo

This notebook walks through the core features of the `Generator` processor.

## 1. Setup

In [1]:
# ! pip install kaggle kagglehub

In [2]:
# Standard
from pathlib import Path

# Third-party
import kagglehub
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel

# Homogene
from homogene import Generator, GenerativeModel

_ = load_dotenv()

Load the [Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews) dataset from Kaggle.

In [3]:
def load_dataset() -> pd.DataFrame:
    """Load the Women's E-Commerce Clothing Reviews dataset from Kaggle.

    Returns:
        `pd.DataFrame` of the dataset.
    """
    dataset_dir = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")
    dataset_path = Path(dataset_dir) / "Womens Clothing E-Commerce Reviews.csv"

    df = pd.read_csv(dataset_path, index_col=0)
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df = df.dropna(subset=["review_text"])

    return df

reviews_df = load_dataset()

reviews_df.head()

,clothing_id,age,title,review_text,rating,recommended_ind,positive_feedback_count,division_name,department_name,class_name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


## 2. Model

Configure the `model` to pass to `Generator`.

**Option 1:** Use a plain string for the simplest case.

In [4]:
model = "gpt-5.5"

**Option 2:** Use a `GenerativeModel` instance for control over parameters like `temperature`, `max_tokens`, etc.

In [5]:
model = GenerativeModel(
    "gpt-5.5",
    temperature=1.0,
    max_tokens=50,
)

## 3. Freeform Generation

In [6]:
reviews_without_title_df = reviews_df[reviews_df["title"].isna()].copy()

print(f"{len(reviews_without_title_df):,} ({len(reviews_without_title_df)/len(reviews_df):.0%}) reviews are missing a title.")

2,966 (13%) reviews are missing a title.


Generate a title for each review that is missing one. `Generator` applies a plain-text instruction to every row **independently** and in **parallel**.

In [7]:
cleaner = Generator(
    df=reviews_without_title_df,
    columns=["review_text", "age"],
    instruction="Generate a title for the review in three words or less.",
    model=model,
)

Use `get_prompt()` to inspect the exact messages sent to the model.

In [8]:
prompt = cleaner.get_prompt(0)

print(prompt)

[
  {
    "role": "system",
    "content": "Generate a title for the review in three words or less."
  },
  {
    "role": "user",
    "content": "{\"review_text\": \"Absolutely wonderful - silky and sexy and comfortable\", \"age\": \"33\"}"
  }
]


Use `run()` to process the `pd.DataFrame`. Optionally, set `n` to limit to a subset first and validate output before committing to the full dataset.

In [9]:
_ = cleaner.run(n=5)

100%|██████████| 5/5 [00:05<00:00,  1.18s/it]

5/5 rows succeeded (100%)


The `result` attribute gives access to the _generation_ and _error_ columns after running.

In [10]:
cleaner.result

,generation,error
0,Silky and Sexy,None
1,Pretty Petite Dress,None
11,Pretty and Flattering,None
30,Cute Flared Pants,None
36,Versatile Work Skirt,None


Increase `n` gradually to validate on larger subsets before running on the full dataset. Optionally, use `save_path` and `save_step` to checkpoint progress to disk periodically.

In [11]:
reviews_df["generated_title"] = cleaner.run(n=20, 
                                            save_path='generated_reviews.csv', 
                                            save_step=10)

100%|██████████| 20/20 [00:08<00:00,  2.45it/s]

20/20 rows succeeded (100%)


A value has now been added to the *'generated_title'* column for every row where *'title'* is missing.

In [12]:
reviews_df[['review_text', 'title', 'generated_title']].head(20)

,review_text,title,generated_title
0,Absolutely wonderful - silky and sexy and comf...,NaN,Silky Sexy Comfort
1,Love this dress! it's sooo pretty. i happene...,NaN,Pretty Petite Find
2,I had such high hopes for this dress and reall...,Some major design flaws,NaN
3,"I love, love, love this jumpsuit. it's fun, fl...",My favorite buy!,NaN
4,This shirt is very flattering to all due to th...,Flattering shirt,NaN
5,"I love tracy reese dresses, but this one is no...",Not for the very petite,NaN
6,I aded this in my basket at hte last mintue to...,Cagrcoal shimmer fun,NaN
7,"I ordered this in carbon for store pick up, an...","Shimmer, surprisingly goes with lots",NaN
8,I love this dress. i usually get an xs but it ...,Flattering,NaN
9,"I'm 5""5' and 125 lbs. i ordered the s petite t...",Such a fun dress!,NaN


## 4. Structured Generation

Define a Pydantic `BaseModel`.

In [ ]:
class Review(BaseModel):
    sentiment: str
    keywords: list[str]

Pass it as `output_schema` to classify the sentiment and extract keywords from every review.

In [14]:
analyzer = Generator(
    df=reviews_df,
    columns=["review_text", "rating"],
    instruction=(
        "Analyze this customer review. Classify the overall sentiment as 'positive', 'neutral', or 'negative'. Extract up to 5 keywords that capture the main topics mentioned."
    ),
    model=model,
    output_schema=Review,
)

analyzer.get_prompt(0)

[
  {
    "role": "system",
    "content": "Analyze this customer review. Classify the overall sentiment as 'positive', 'neutral', or 'negative'. Extract up to 5 keywords that capture the main topics mentioned."
  },
  {
    "role": "user",
    "content": "{\"review_text\": \"Absolutely wonderful - silky and sexy and comfortable\", \"rating\": \"4\"}"
  }
]

In [15]:
_ = analyzer.run(n=20)

100%|██████████| 20/20 [00:04<00:00,  4.12it/s]

20/20 rows succeeded (100%)


In [16]:
analyzer.result.head()

,generation,error
0,"sentiment='positive' keywords=['wonderful', 's...",None
1,"sentiment='positive' keywords=['dress', 'prett...",None
2,"sentiment='negative' keywords=['sizing', 'fit'...",None
3,"sentiment='positive' keywords=['jumpsuit', 'fu...",None
4,"sentiment='positive' keywords=['flattering', '...",None


Expand the fields in the `Review` column into separate `pd.DataFrame` columns.

In [17]:
reviews_df["sentiment"] = analyzer.result['generation'].map(lambda x: x.sentiment if x is not None else None)
reviews_df["keywords"]  = analyzer.result['generation'].map(lambda x: x.keywords if x is not None else None)

reviews_df[["review_text", "rating", "sentiment", "keywords"]].head()

,review_text,rating,sentiment,keywords
0,Absolutely wonderful - silky and sexy and comf...,4,positive,"[wonderful, silky, sexy, comfortable]"
1,Love this dress! it's sooo pretty. i happene...,5,positive,"[dress, pretty, petite, length, midi]"
2,I had such high hopes for this dress and reall...,3,negative,"[sizing, fit, tight underlayer, cheap net laye..."
3,"I love, love, love this jumpsuit. it's fun, fl...",5,positive,"[jumpsuit, fun, flirty, fabulous, compliments]"
4,This shirt is very flattering to all due to th...,5,positive,"[flattering, adjustable front tie, perfect len..."


## 5. Error Handling

Demonstrate both error modes using an invalid API key.

In [18]:
faulty_model = GenerativeModel("gpt-5.5", api_key="invalid-key")

faulty = Generator(
    df=reviews_df.head(),
    columns=["review_text"],
    instruction="Write a title for this review in three words or less.",
    model=faulty_model,
)

**Case 1:** `on_error = "none"` <span style="color: grey;">_(default)_</span>

Errors are silently captured. Failed rows store `None` in the _generation_ column and the error message in the _error_ column. The run never crashes.

In [19]:
try:
    faulty.run(n=5, on_error="none")
    faulty.result.head()

except Exception as e:
    print(f"Error: {e}")

100%|██████████| 5/5 [00:00<00:00,  8.96it/s]

0/5 rows succeeded (0%)


**Case 2:** `on_error = "raise"`

The first exception is raised immediately, stopping the run.

In [20]:
try:
    faulty.run(n=5, on_error="raise")
    faulty.result.head()

except Exception as e:
    print(f"Error: {e}")


  0%|          | 0/5 [00:00<?, ?it/s]

Error: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: invalid-key. You can find your API key at https://platform.openai.com/account/api-keys.


## 6. Other Use Cases

Other use cases for `Generator`:
- Extract structured information from unstructured text
- Run an LLM-as-a-judge on each row